[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JulesMalin/isba2411-nlp/blob/main/Week%205/L10_Demo_FindingThemes.ipynb)

# ISBA 2411 · Week 5 · Lecture 10
## Finding the Themes in the Pile: Topic Modeling with Embeddings

Last time you *told* the model what to look for: you picked the categories (the zero-shot Router) and asked the questions (QA). Tonight you hand it **nothing**. We drop ~5,000 unlabeled Steam reviews in and let the **themes fall out on their own**.

**The whole idea in one line:** turn each review into a *point in meaning-space*, let the computer *find the clusters*, then **you name them**.

<img src="https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/Week%205/diagrams/journey/pipeline.png?v=8" width="780" alt="pipeline"/>

*The five stages, end to end. `BERTopic` runs the first four; the last one is yours. We build this up one stage at a time below.*

We do it twice:
- **Part A: by hand, on five reviews.** Slow enough to watch every step: embed → reduce → cluster → read → name.
- **Part B: one button, on all 5,000.** The exact same idea, at scale, with `BERTopic`.
- **Part C: your turn.** *Name That Topic*: six real topics from your own run, and you name them.

> 🎓 **Class connection.** The embedding model here is the same *pretrained transformer* family you met on Hugging Face in L9, now pointed at an **unsupervised** problem.

> 🚀 **Final project.** “Group my text by theme” is a milestone many projects need. Swap the Steam reviews for your corpus and this notebook is your template.

---
**Runtime:** Colab, CPU is fine. The first install + embedding pass takes a couple of minutes; everything after is instant.

## How to run this tonight

Sit with your **final project team** (the same teams as Canvas). **One laptop per team** is enough, two is fine. Do this **now**, before anything else:

1. **Runtime → Run all.** It will churn for about 4 minutes (install, then embedding 5,000 reviews).
2. Ignore it and listen. By the time we reach Stage 1 it will be done, and you will have your own results to look at.
3. **Open the worksheet in a second tab:** **the shared Google Sheet** (link posted on Canvas). That is where your answers go at the end. Sign in with your **SCU account**.
4. **Do not touch the “Going further” section at the bottom** until we get there. Those cells are commented out on purpose, one of them shows you the answer to tonight’s exercise.

> **Everyone gets the same topics.** The seed is pinned, so your six cards are the same six cards as the team next to you. That is what makes the comparison at the end work. If your cards look different from the projector, say so.

## 0 · Setup

**What this cell does.** Installs `bertopic` (which brings `sentence-transformers`, `umap-learn`, and `hdbscan` with it) plus the usual data tools, then imports what we need. One install line; no version pins to fight with.

In [ ]:
%pip install -q bertopic sentence-transformers

import re, html, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer, util
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.feature_extraction.text import CountVectorizer

plt.rcParams['figure.dpi'] = 120
print('ready')

# Part A · Five reviews, by hand

<img src="https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/Week%205/diagrams/journey/five_chips.png?v=8" width="620" alt="five_chips"/>

Small enough to follow with your eyes. Two themes are obvious, **and one review is a troublemaker.** Read them and make two predictions before we run anything: *how many themes?* and *which review won’t fit?*

*(The pictures below are the lecture slides: this notebook and the deck are the same journey, step for step.)*

In [ ]:
REVIEWS = [
    'Amazing story.',              # 1  story
    'Great characters.',           # 2  story
    'Crashes constantly.',         # 3  bugs
    'Full of bugs.',               # 4  bugs
    'Good story, but it crashes.', # 5  <- the in-between one
]
for i, r in enumerate(REVIEWS, 1):
    print(f'{i}. {r}')

### Stage 1 · Embed  ·  words become numbers

<img src="https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/Week%205/diagrams/journey/embed.png?v=8" width="620" alt="embed"/>

**What this cell does.** Loads a small pretrained embedding model and turns each review into a list of **384 numbers**, its coordinates in “meaning-space.” You can’t read those numbers, and you don’t need to. Only one property matters, and we check it next.

> **Under the hood.** `all-MiniLM-L6-v2` is a 6-layer MiniLM sentence-transformer that mean-pools token vectors into one **384-dim** vector per text; we compare vectors with **cosine similarity**. Ref: Reimers & Gurevych, *Sentence-BERT*, EMNLP 2019 (arXiv:1908.10084).

In [ ]:
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
emb = embed_model.encode(REVIEWS)
print('each review is now a point with', emb.shape[1], 'coordinates')
print('shape:', emb.shape)   # (5 reviews, 384 numbers each)

### The one property that matters: similar meaning → nearby points

**What this cell does.** Measures how close two reviews are in meaning-space (cosine similarity: 1.0 = identical direction, 0 = unrelated). Watch review **1 vs 2** (both about story) against **1 vs 3** (story vs crashes).

> The model was **never told the topics.** It figured out that 1 and 2 belong together purely from the words, that’s the whole magic.

In [ ]:
def sim(i, j):
    return float(util.cos_sim(emb[i-1], emb[j-1]))

print(f'1 vs 2  (story vs story)   = {sim(1,2):.2f}   <- high, same theme')
print(f'1 vs 3  (story vs crashes) = {sim(1,3):.2f}   <- low, different theme')
print(f'3 vs 4  (crashes vs bugs)  = {sim(3,4):.2f}   <- high, same theme')
print(f'1 vs 5  (story vs #5)      = {sim(1,5):.2f}')
print(f'3 vs 5  (crashes vs #5)    = {sim(3,5):.2f}   <- #5 leans to both')

### Stage 1 · Embed  ·  similar meaning lands nearby

<img src="https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/Week%205/diagrams/journey/points.png?v=8" width="620" alt="points"/>

**What this cell does.** 384 numbers is too many to plot. We **reduce** each review to just 2 numbers, an `(x, y)`, keeping the far-apart things far apart. Here we use PCA (deterministic, no settings); at scale in Part B, BERTopic uses **UMAP** for the same job.

> 🎓 **Class connection.** This is the *dimension reduction* half of tonight’s syllabus line, shown, not lectured. “Squash many numbers down to a few, keep the shape.”

> **Under the hood.** PCA is a linear projection. BERTopic instead uses **UMAP** (`n_neighbors=15, n_components=5, metric='cosine'`), which preserves local neighborhoods better on large corpora. Ref: McInnes, Healy & Melville, *UMAP*, 2018 (arXiv:1802.03426).

In [ ]:
xy = PCA(n_components=2, random_state=0).fit_transform(emb)
for i, (x, y) in enumerate(xy, 1):
    print(f'review {i}:  x={x:+.2f}  y={y:+.2f}')

### Step 3 · Plot the dots

**What this cell does.** Puts the five reviews on a map. Two clusters should jump out, and review **5** should be sitting out in the gap between them.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.2))
ax.scatter(xy[:,0], xy[:,1], s=260, c='#B4B2A9', edgecolors='#2C2C2A', zorder=3)
for i, (x, y) in enumerate(xy, 1):
    ax.annotate(str(i), (x, y), ha='center', va='center', fontweight='bold', zorder=4)
ax.set_title('Five reviews in meaning-space (2-D)')
ax.set_xticks([]); ax.set_yticks([])
for s in ax.spines.values(): s.set_visible(False)
plt.tight_layout(); plt.show()

### Stage 2 · Cluster  ·  circle the clusters

<img src="https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/Week%205/diagrams/journey/clusters.png?v=8" width="620" alt="clusters"/>

**What this cell does.** We can *see* two clusters, so we ask the computer for two and let it assign every review. Each review also gets a **fit score**, how much closer it sits to its own cluster than to the other (100% = firmly inside one theme, near 0% = torn between both). The lowest-fit review is our **outlier**: the algorithm can file it, but not *confidently*.

> **Under the hood.** BERTopic clusters with **HDBSCAN**, a density method that discovers the number of clusters itself and labels sparse points as noise (`-1`). Five points is too few for HDBSCAN, so our toy uses `KMeans(2)` plus a fit-margin. Ref: Campello, Moulavi & Sander, PAKDD 2013; McInnes et al., *hdbscan*, JOSS 2017.

> Prediction check: which review has the weakest fit?
>
> *In Part B, on 5,000 reviews, you can’t eyeball how many clusters there are, so the real algorithm (HDBSCAN, inside BERTopic) figures out the number itself and drops the torn cases into an ‘outlier’ bucket.*

In [ ]:
emb_n = normalize(emb)                  # compare by direction (meaning), not length
km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(emb_n)
labels = km.labels_

dist = km.transform(emb_n)             # distance from each review to each cluster center
near = np.sort(dist, axis=1)
fit  = 1 - near[:, 0] / near[:, 1]      # ~1 = firmly in one theme, ~0 = torn between both
outlier = int(fit.argmin())            # the single weakest-fit review

for i, (lab, f) in enumerate(zip(labels, fit), 1):
    flag = '   <- OUTLIER (weakest fit)' if i - 1 == outlier else ''
    print(f'review {i}: cluster {lab}   fit {f:.0%}{flag}')

palette = {0:'#1D9E75', 1:'#7F77DD'}
fig, ax = plt.subplots(figsize=(6, 4.2))
for i, ((x, y), lab) in enumerate(zip(xy, labels)):
    if i == outlier:
        ax.scatter(x, y, s=260, facecolors='none', edgecolors='#A32D2D', linewidths=2, zorder=3)
    else:
        ax.scatter(x, y, s=260, c=palette.get(lab, '#888780'), edgecolors='#2C2C2A', zorder=3)
    ax.annotate(str(i + 1), (x, y), ha='center', va='center', fontweight='bold',
                color='#A32D2D' if i == outlier else 'white', zorder=4)
ax.set_title('Two clusters + the weakest-fit outlier (hollow)')
ax.set_xticks([]); ax.set_yticks([])
for s in ax.spines.values(): s.set_visible(False)
plt.tight_layout(); plt.show()

### Stage 3 · Name  ·  read each cluster (top words)

<img src="https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/Week%205/diagrams/journey/read.png?v=8" width="620" alt="read"/>

**What this cell does.** For each cluster, counts the most distinctive words, using only the confident members. This bag of words is the *only* thing the algorithm gives you about a topic. **Notice what’s missing: a name.**

> **Under the hood.** Our toy uses a plain word count. BERTopic scores words with **c-TF-IDF** (class-based TF-IDF): treat a whole cluster as one document, then surface the terms that most distinguish it from the other clusters. Ref: Grootendorst, *BERTopic*, 2022 (arXiv:2203.05794).

In [ ]:
def top_words(texts, n=6):
    cv = CountVectorizer(stop_words='english')
    X = cv.fit_transform(texts)
    counts = np.asarray(X.sum(axis=0)).ravel()
    vocab = cv.get_feature_names_out()
    return [vocab[i] for i in counts.argsort()[::-1][:n]]

core = [i != outlier for i in range(len(REVIEWS))]
for lab in sorted(set(labels)):
    members = [REVIEWS[i] for i in range(len(REVIEWS)) if labels[i] == lab and core[i]]
    print(f'cluster {lab} top words -> {top_words(members)}')
print(f'outlier (you decide what to do): review {outlier + 1} -> "{REVIEWS[outlier]}"')

### Stage 3 · Name  ·  *this part is yours*

<img src="https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/Week%205/diagrams/journey/name.png?v=8" width="620" alt="name"/>

The algorithm handed you `['story', 'amazing', 'characters', ...]` and `['crashes', 'bugs', ...]`. It will **never** write “Story” or “Bugs”, a human reads the words and the examples and names the theme.

| Cluster | Top words | **Your name** |
|---|---|---|
| cluster 0 | story, amazing, characters… | ? |
| cluster 1 | crashes, bugs, constantly… | ? |
| outlier (#5) | (none) | keep? drop? send to review? |

> **Read the words, then commit to a name.** The clustering is mechanical. Calling cluster 1 “Crashes & bugs” instead of “Performance” changes what a team does next, so the label is the part that carries the risk.

### The catch

<img src="https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/Week%205/diagrams/journey/catch.png?v=8" width="620" alt="catch"/>

Review 5 belongs to **both** themes, so really to **neither**. Two honest limits to carry forward:
- **Outliers.** Some texts fit no clean topic; at scale they land in HDBSCAN’s `-1` bucket. You decide what to do with them.
- **Coherent is not correct.** A tidy word-list can still be a junk grouping, and *how many* topics you get is a dial you set. Your judgment is the safeguard.

# Part B · One button, on all 5,000 reviews

<img src="https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/Week%205/diagrams/journey/scale.png?v=8" width="620" alt="scale"/>

Everything in Part A (embed, reduce, cluster, top words) is exactly what `BERTopic` does in **one call**. We just did it slowly by hand so this next cell isn’t magic.

> **Under the hood.** `BERTopic().fit_transform(docs)` chains the four steps end to end: **SBERT** embed → **UMAP** reduce → **HDBSCAN** cluster → **c-TF-IDF** words. Ref: Grootendorst, *BERTopic*, 2022 (arXiv:2203.05794).

## 1 · Load the Steam reviews

**What this cell does.** Reads the ~5,000 English reviews you already met in L9 and tidies the text.

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/Week%205/steam_reviews_demo.csv'
df = pd.read_csv(DATA_URL)
df['review'] = df['review'].map(lambda t: re.sub(r'\s+', ' ', html.unescape(str(t))).strip())
df = df[df['review'].str.len() > 40].reset_index(drop=True)
docs = df['review'].tolist()
print(f'{len(docs):,} reviews across {df.game.nunique()} games')

## 2 · Embed once, then fit the model

**What this cell does.** Encodes all reviews into meaning-space **once** (the slow part, ~1–2 min on CPU) and hands those vectors to `BERTopic`, so re-running topic settings later is fast. `BERTopic` then runs the same three moves from Part A: reduce (UMAP) → cluster (HDBSCAN) → top words per cluster.

> 🚀 **Final project.** Caching embeddings once and reusing them is the standard trick, the embedding pass is the expensive step.

> **Under the hood.** BERTopic’s default vectorizer **keeps stopwords**, so topics come back looking like `0_story_amazing_the_game` and `2_crashes_to_fix_it`. Passing `CountVectorizer(stop_words='english')` strips them, and the top words become readable enough to actually name. Do this on any real corpus.

> ⚠️ **Why the seed matters.** UMAP is stochastic and BERTopic does not pin it for you, so two runs of this cell would find **different topics**: your card C would not be anyone else’s card C, and tonight’s whole compare-the-names exercise would fall apart. `random_state=42` makes the run reproducible. (The cost: UMAP drops to a single thread, so it is a little slower.)

In [ ]:
from bertopic import BERTopic
from umap import UMAP

embeddings = embed_model.encode(docs, show_progress_bar=True)

# strip English stopwords, or every topic's top words come back as 'the, and, game, it'
vectorizer = CountVectorizer(stop_words='english')

# PIN THE SEED. UMAP is random by default, so without this every run finds different
# topics and nobody's 'card C' is the same card C.
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                  metric='cosine', random_state=42)

topic_model = BERTopic(embedding_model=embed_model, umap_model=umap_model, min_topic_size=30,
                       vectorizer_model=vectorizer, verbose=False)
topics, probs = topic_model.fit_transform(docs, embeddings)
print('done, found', len(set(topics)) - (1 if -1 in topics else 0), 'topics (plus an outlier bucket)')

## 3 · What did it find?

**What this cell does.** Lists the topics by size. Topic **-1** is the outlier bucket, the big-corpus version of review #5. `Name` is BERTopic’s auto-guess from the top words; treat it as a *starting point*, not the final label.

In [ ]:
info = topic_model.get_topic_info()
info[['Topic', 'Count', 'Name']].head(12)

### The top words behind a few topics

**What this cell does.** Prints the word-lists, the same ‘Step 5’ output as the toy, just more of them. Read them and practice naming in your head.

In [ ]:
for t in [0, 1, 2, 3, 4]:
    words = ', '.join(w for w, _ in topic_model.get_topic(t)[:8])
    print(f'Topic {t}: {words}')

### The most representative reviews for one topic

**What this cell does.** Pulls the reviews closest to a topic’s center, the evidence you’d use to justify a name.

In [ ]:
for r in topic_model.get_representative_docs(0)[:3]:
    print('•', r[:160], '...')

### See the whole map

**What this cell does.** The full-corpus version of the Part-A scatter: every topic as a bubble in 2-D. Hover to read the words. **This one picture is topic modeling + dimension reduction together.**

In [ ]:
topic_model.visualize_topics()

# Part C · Your turn: *Name That Topic*

<img src="https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/Week%205/diagrams/journey/mystery.png?v=8" width="620" alt="mystery"/>

**Run the cell below.** It prints **your six cards (A to F)**: each is a real topic from the run you just did, shown only by its top words and three example reviews. There are no names on them, because the model does not make names. That is your job.

In [ ]:
import textwrap
def show_topic_card(letter, tid):
    words = ', '.join(w for w, _ in topic_model.get_topic(tid)[:10])
    print(f'=== Topic {letter} ===')
    print('top words:', words)
    try:
        docs = topic_model.get_representative_docs(tid) or []
    except Exception:
        docs = []
    for r in docs[:3]:
        print('   •', textwrap.shorten(r, 150))
    print()

# 5 real topics + the -1 outlier bucket as the built-in 'junk' card
cards = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': -1}
for letter, tid in cards.items():
    show_topic_card(letter, tid)

---
## 📋 The deliverable

**Where:** **the shared Google Sheet** (link posted on Canvas). Find **your final project team's row** and fill in that row only. No names needed, just your team number. **Not in this notebook.**

**You are done when your row has all three:**

| # | What | What it looks like |
|---|---|---|
| **1** | **Six names** | a short, specific name for each card A to F |
| **2** | **The junk call** | which card is *not* a real theme, and how you could tell |
| **3** | **One mis-file** | a review sitting in the wrong card: quote it, and say where it belongs |

**Time:** about 20 minutes. **Scored for taking part, not for being right.** The Doc has a worked example at the top showing a weak name next to a good one, if you want the shape first.

> 🎓 This is the *Analysis & Insight* muscle the midterm grades. Tonight is the rehearsal.

> ⚠️ **Stop here** until your instructor says otherwise. Everything below is optional, and one of those cells shows you a machine's answer to the exercise you are in the middle of.

## Going further (optional)

Finished early, or want to keep poking after class? Pick one. **These cells are commented out** so that *Run all* does not fire them: uncomment the one you want.

### Mid · change the granularity dial

**What this cell does.** Re-runs BERTopic with a smaller `min_topic_size`, so it keeps finer topics. Compare the count and names to the first run, and note what **splits** into two and what **merges**. It refits the model, so give it a minute.

In [ ]:
# Uncomment to run (about a minute: it refits the model).

# tm_fine = BERTopic(embedding_model=embed_model, umap_model=umap_model, min_topic_size=15,
#                    vectorizer_model=vectorizer, verbose=False)
# topics_fine, _ = tm_fine.fit_transform(docs, embeddings)
# n_first = len(set(topics)) - (1 if -1 in topics else 0)
# n_fine  = len(set(topics_fine)) - (1 if -1 in topics_fine else 0)
# print(f'min_topic_size=30 -> {n_first} topics   |   min_topic_size=15 -> {n_fine} topics')
# tm_fine.get_topic_info()[['Topic', 'Count', 'Name']].head(15)

### Advanced · defend a name with evidence

**What this cell does.** Prints a topic’s top words plus its **3 most representative reviews**, the evidence you’d cite to justify the name you gave it. Change `tid` to any topic number.

In [ ]:
# Uncomment to run (instant).

# tid = 0   # <- pick any topic id from the table above
# print('top words:', ', '.join(w for w, _ in topic_model.get_topic(tid)[:10]))
# print()
# for r in topic_model.get_representative_docs(tid)[:3]:
#     print('•', r[:220])

### Automated · let an LLM write the names

**What this cell does.** Swaps in a **representation model**: BERTopic hands each topic’s keywords and example reviews to a small LLM (`flan-t5-base`, free and local in Colab) and asks it for a label. This is the automated version of the one human step.

> **The point.** Automating the name doesn’t remove the judgment, it **moves** it. The LLM returns a fluent, confident label, and a *wrong* one is now **harder** to catch because it reads authoritative. Compare its labels with the names you wrote on your worksheet: **which ones did it get wrong?**

> **Under the hood.** `TextGeneration` templates each topic’s `[KEYWORDS]` and `[DOCUMENTS]` into a prompt. Swap it for `OpenAI`, `LiteLLM`, `LlamaCPP`, or `Cohere` to use a bigger model. Ref: Grootendorst, *BERTopic* (representation models), 2022.

> ⚠️ Heads up: this **overwrites the labels in place**, so re-running the Part C card cell afterwards will show the LLM’s labels instead of blanks.

In [ ]:
# SPOILER: do not run until you have written your own six names in the Doc.
# Uncomment everything below, then run.

# from transformers import pipeline
# from bertopic.representation import TextGeneration

# prompt = ('I have a topic described by these keywords: [KEYWORDS]. '
#           'Here are a few of its reviews: [DOCUMENTS]. '
#           'Give a short, specific topic label:')

# generator = pipeline('text2text-generation', model='google/flan-t5-base')
# rep = TextGeneration(generator, prompt=prompt)

# # keep the keyword view before the LLM overwrites it
# before = {L: ', '.join(w for w, _ in topic_model.get_topic(t)[:5]) for L, t in cards.items()}

# # same topics, same ids: only the labelling step is redone
# topic_model.update_topics(docs, representation_model=rep)

# for L, t in cards.items():
#     llm_label = topic_model.get_topic(t)[0][0]
#     print(f'card {L}   LLM label: {llm_label!r}')
#     print(f'          you saw: {before[L]}')
#     print()

#### Optional: a stronger labeller (needs a GPU runtime)

`flan-t5-base` above is the default on purpose: it always runs, no GPU, no API key. But its labels are weak, and that quietly *undersells* tonight’s point. A **bad** label is easy to catch; the lesson is that a **good** label is the one that slips past you. For the sharper version, switch **Runtime → Change runtime type → T4 GPU** and run the cell below.

**Zephyr-7B-beta** (Apache 2.0, ungated) is a Mistral-class model: 4-bit quantized it gives genuinely good labels in about a second each on a T4. Meta’s Llama-3.x repos are **gated** (every student would need to accept the licence and make an HF token), so we avoid those in class.

> The cell is **commented out on purpose** so it can never break a live session. Uncomment it only after switching the runtime to a GPU.

> ⚠️ Two gotchas worth knowing. BERTopic’s own docs still show `CMAKE_ARGS="-DLLAMA_CUBLAS=on"`, which is **deprecated** (it is `-DGGML_CUDA=on` now); the prebuilt wheel below skips the compile *and* the flag. And `Llama.from_pretrained` pulls the GGUF straight from the Hub, so there is no manual 4.4 GB `wget`.

In [ ]:
# OPTIONAL: stronger labels with Zephyr-7B. Runtime -> Change runtime type -> T4 GPU first.
# Uncomment everything below to run it.

# !pip install -q llama-cpp-python \
#     --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121

# from llama_cpp import Llama
# from bertopic.representation import LlamaCPP

# llm = Llama.from_pretrained(
#     repo_id='TheBloke/zephyr-7B-beta-GGUF',
#     filename='*Q4_K_M.gguf',      # ~4.4 GB, cached after the first pull
#     n_ctx=4096, n_gpu_layers=-1,  # -1 = offload the whole model to the GPU
#     stop='\n', verbose=False)

# topic_model.update_topics(docs, representation_model=LlamaCPP(llm))
# for L, t in cards.items():
#     print(f'card {L}   Zephyr: {topic_model.get_topic(t)[0][0]!r}')

---
### Wrap
BERTopic grouped the 5,000 reviews and printed a word-list for each cluster. Your work tonight was reading those lists, dropping the incoherent ones, and writing topic names a product team could act on.

> 🚀 **Next week.** Those same meaning-space points become the engine of **retrieval (RAG)**, same embeddings, new job.

## References & tools

**Papers**
- Reimers, N. & Gurevych, I. (2019). *Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks.* EMNLP. arXiv:1908.10084.
- Wang, W. et al. (2020). *MiniLM: Deep Self-Attention Distillation for Task-Agnostic Compression.* NeurIPS. arXiv:2002.10957.
- McInnes, L., Healy, J. & Melville, J. (2018). *UMAP: Uniform Manifold Approximation and Projection.* arXiv:1802.03426.
- Campello, R., Moulavi, D. & Sander, J. (2013). *Density-Based Clustering Based on Hierarchical Density Estimates.* PAKDD.
- Grootendorst, M. (2022). *BERTopic: Neural topic modeling with a class-based TF-IDF procedure.* arXiv:2203.05794.

**Tools:** `bertopic` · `sentence-transformers` (`all-MiniLM-L6-v2`) · `umap-learn` · `hdbscan` · `scikit-learn`.

**Textbook:** Jurafsky & Martin, *Speech and Language Processing* (3rd ed.), Ch. 6: Vector Semantics & Embeddings.